# Manufacturing Quality Analytics
A guided exploration of a fully synthetic quality dataset.


## 1. Load libraries and data


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = ROOT / 'data' / 'raw' / 'synthetic_quality_data.csv'
df = pd.read_csv(data_path, parse_dates=['inspection_date'])
df.head()


## 2. Check data quality


In [ ]:
print('Shape:', df.shape)
display(df.isna().sum().to_frame('missing_values'))
assert (df['units_inspected'] > 0).all()
assert (df['defect_units'] <= df['units_inspected']).all()


## 3. Calculate headline KPIs


In [ ]:
total_inspected = df['units_inspected'].sum()
total_defects = df['defect_units'].sum()
overall_defect_rate = total_defects / total_inspected
print(f'Inspection records: {len(df):,}')
print(f'Units inspected: {total_inspected:,}')
print(f'Defective units: {total_defects:,}')
print(f'Overall defect rate: {overall_defect_rate:.2%}')


## 4. Monthly defect-rate trend


In [ ]:
df['month'] = df['inspection_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month', as_index=False).agg(
    units_inspected=('units_inspected', 'sum'),
    defect_units=('defect_units', 'sum')
)
monthly['defect_rate'] = monthly['defect_units'] / monthly['units_inspected']
ax = monthly.plot(x='month', y='defect_rate', marker='o', legend=False,
                  title='Monthly Defect Rate')
ax.set_ylabel('Defect rate')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()


## 5. Compare suppliers
Compare rates alongside inspected volume; a rate based on a small volume may be less stable.


In [ ]:
supplier_summary = df.groupby('supplier', as_index=False).agg(
    units_inspected=('units_inspected', 'sum'),
    defect_units=('defect_units', 'sum')
)
supplier_summary['defect_rate'] = (
    supplier_summary['defect_units'] / supplier_summary['units_inspected']
)
supplier_summary.sort_values('defect_rate', ascending=False)


## 6. Defect categories and interpretation


In [ ]:
category_summary = (df.groupby('defect_category', as_index=False)
                    .agg(defect_units=('defect_units', 'sum'))
                    .sort_values('defect_units', ascending=False))
category_summary


## 7. Write your conclusions
Use this section to record three evidence-based observations, one limitation, and one recommendation. Avoid treating synthetic results as real-world findings.
